<a href="https://colab.research.google.com/github/Yernar8121/Gold-Price-ML-Project/blob/main/demo_app_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio

In [2]:
import pandas as pd
import numpy as np
import gradio as gr
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv("/content/XAU_1d_data.csv", sep=";")

df.head()

,Date,Open,High,Low,Close,Volume
0,2004.06.11 00:00,384.0,384.8,382.8,384.1,272
1,2004.06.14 00:00,384.3,385.8,381.8,382.8,1902
2,2004.06.15 00:00,382.8,388.8,381.1,388.6,1951
3,2004.06.16 00:00,387.1,389.8,382.6,383.8,2014
4,2004.06.17 00:00,383.6,389.3,383.0,387.6,1568


In [4]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

df["Price_Range"] = df["High"] - df["Low"]
df["Open_Close_Diff"] = df["Close"] - df["Open"]
df["High_Low_Ratio"] = df["High"] / df["Low"]
df["Daily_Return"] = df["Close"].pct_change()

df["Close_Lag_1"] = df["Close"].shift(1)
df["Close_Lag_3"] = df["Close"].shift(3)
df["Close_Lag_7"] = df["Close"].shift(7)


df["MA_7"] = df["Close"].rolling(window=7).mean()
df["MA_14"] = df["Close"].rolling(window=14).mean()
df["MA_30"] = df["Close"].rolling(window=30).mean()

df["Volatility_7"] = df["Close"].rolling(window=7).std()
df["Volatility_14"] = df["Close"].rolling(window=14).std()

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek



df["Target_Delta"] = df["Close"].shift(-1) - df["Close"]

df["Target_Direction"] = (df["Target_Delta"] >= 0).astype(int)

df["Naive_Delta"] = 0

today_delta = df["Close"] - df["Close"].shift(1)
df["Naive_Direction"] = (today_delta >= 0).astype(int)

df = df.dropna()

print("Preprocessing and feature engineering completed!")
df.head()

Preprocessing and feature engineering completed!


,Date,Open,High,Low,Close,Volume,Price_Range,Open_Close_Diff,High_Low_Ratio,Daily_Return,...,Volatility_7,Volatility_14,Year,Month,Day,DayOfWeek,Target_Delta,Target_Direction,Naive_Delta,Naive_Direction
29,2004-07-23,394.3,395.0,386.5,389.3,1899,8.5,-5.0,1.021992,-0.010170,...,6.312082,6.023493,2004,7,23,4,0.5,1,0,0
30,2004-07-26,389.3,392.1,388.1,389.8,1992,4.0,0.5,1.010307,0.001284,...,6.874037,6.314087,2004,7,26,0,-3.2,0,0,1
31,2004-07-27,390.5,393.1,385.1,386.6,3054,8.0,-3.9,1.020774,-0.008209,...,6.598340,7.382397,2004,7,27,1,0.7,1,0,0
32,2004-07-28,386.1,389.6,384.5,387.3,3205,5.1,1.2,1.013264,0.001811,...,5.151514,7.768265,2004,7,28,2,-0.2,0,0,1
33,2004-07-29,387.3,390.8,385.3,387.1,2521,5.5,-0.2,1.014275,-0.000516,...,3.555613,7.900118,2004,7,29,3,3.4,1,0,0


In [5]:
FEATURES = [
    "Price_Range",
    "Open_Close_Diff",
    "High_Low_Ratio",
    "Daily_Return",

    "Close_Lag_1",
    "Close_Lag_3",
    "Close_Lag_7",

    "MA_7",
    "MA_14",
    "MA_30",

    "Volatility_7",
    "Volatility_14",

    "Year",
    "Month",
    "Day",
    "DayOfWeek",

    "Volume"
]

features = [f for f in FEATURES if f in df.columns]

X = df[features]
y_delta = df["Target_Delta"]
y_dir = df["Target_Direction"]

X_train, X_test, y_delta_train, y_delta_test, y_dir_train, y_dir_test = train_test_split(
    X,
    y_delta,
    y_dir,
    test_size=0.2,
    shuffle=False
)

print("Features used:")
print(features)

Features used:
['Price_Range', 'Open_Close_Diff', 'High_Low_Ratio', 'Daily_Return', 'Close_Lag_1', 'Close_Lag_3', 'Close_Lag_7', 'MA_7', 'MA_14', 'MA_30', 'Volatility_7', 'Volatility_14', 'Year', 'Month', 'Day', 'DayOfWeek', 'Volume']


In [6]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

In [7]:
clf_direction = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=0.01,
        penalty="l2",
        solver="lbfgs",
        max_iter=1000
    ))
])

clf_direction.fit(X_train, y_dir_train)

y_dir_pred = clf_direction.predict(X_test)

direction_accuracy = accuracy_score(y_dir_test, y_dir_pred)

print("Direction Classification Accuracy:", direction_accuracy)
print("Accuracy %:", direction_accuracy * 100)
print(classification_report(y_dir_test, y_dir_pred))
print(confusion_matrix(y_dir_test, y_dir_pred))

reg_delta = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=1.0))
])

reg_delta.fit(X_train, y_delta_train)

y_delta_pred = reg_delta.predict(X_test)

delta_mae = mean_absolute_error(y_delta_test, y_delta_pred)
delta_rmse = np.sqrt(mean_squared_error(y_delta_test, y_delta_pred))
delta_r2 = r2_score(y_delta_test, y_delta_pred)

print("RIDGE REGRESSION DELTA")
print("MAE:", delta_mae)
print("RMSE:", delta_rmse)
print("R2:", delta_r2)

Direction Classification Accuracy: 0.5031789282470481
Accuracy %: 50.31789282470481
              precision    recall  f1-score   support

           0       0.45      0.36      0.40       510
           1       0.53      0.63      0.58       591

    accuracy                           0.50      1101
   macro avg       0.49      0.49      0.49      1101
weighted avg       0.50      0.50      0.49      1101

[[182 328]
 [219 372]]
RIDGE REGRESSION DELTA
MAE: 19.595633902914877
RMSE: 37.86338303355801
R2: -0.028190544109560678


In [8]:
naive_delta = np.zeros(len(y_delta_test))

naive_delta_mae = mean_absolute_error(y_delta_test, naive_delta)
naive_delta_rmse = np.sqrt(mean_squared_error(y_delta_test, naive_delta))
naive_delta_r2 = r2_score(y_delta_test, naive_delta)

print("NAIVE BASELINE DELTA")
print("MAE:", naive_delta_mae)
print("RMSE:", naive_delta_rmse)
print("R2:", naive_delta_r2)

NAIVE BASELINE DELTA
MAE: 19.211689373297006
RMSE: 37.446426939750104
R2: -0.005670115440362444


In [9]:
def make_price_chart():
    chart_df = df.copy()

    chart_df["Date"] = pd.to_datetime(chart_df["Date"])
    chart_df["Year"] = chart_df["Date"].dt.year

    yearly_avg = chart_df.groupby("Year", as_index=False)["Close"].mean()

    fig = px.bar(
        yearly_avg,
        x="Year",
        y="Close",
        title="Average XAU Close Price by Year",
        labels={
            "Year": "Year",
            "Close": "Average Close Price"
        },
        text=yearly_avg["Close"].round(2)
    )

    fig.update_layout(
        height=360,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="white"),
        title_font=dict(size=22, color="white"),
        xaxis=dict(
            title="Year",
            gridcolor="rgba(255,255,255,0.15)"
        ),
        yaxis=dict(
            title="Average Close Price",
            gridcolor="rgba(255,255,255,0.15)"
        ),
        bargap=0.25
    )

    fig.update_traces(
        textposition="outside",
        hovertemplate=
        "Year: %{x}<br>" +
        "Average Close Price: %{y:.2f}<extra></extra>"
    )

    return fig

In [10]:
def create_input_features(
    Prev_Open, Prev_High, Prev_Low, Prev_Close, Prev_Volume,
    Open, High, Low, Close, Volume
):
    last_row = df.iloc[-1]

    input_data = pd.DataFrame([{
        "Price_Range": High - Low,
        "Open_Close_Diff": Close - Open,
        "High_Low_Ratio": High / Low,
        "Daily_Return": (Close - Prev_Close) / Prev_Close,

        "Close_Lag_1": Prev_Close,
        "Close_Lag_3": last_row["Close_Lag_3"],
        "Close_Lag_7": last_row["Close_Lag_7"],

        "MA_7": last_row["MA_7"],
        "MA_14": last_row["MA_14"],
        "MA_30": last_row["MA_30"],

        "Volatility_7": last_row["Volatility_7"],
        "Volatility_14": last_row["Volatility_14"],

        "Year": last_row["Year"],
        "Month": last_row["Month"],
        "Day": last_row["Day"],
        "DayOfWeek": last_row["DayOfWeek"],

        "Volume": Volume
    }])

    return input_data[features]

In [11]:
def predict_app(
    Prev_Open, Prev_High, Prev_Low, Prev_Close, Prev_Volume,
    Open, High, Low, Close, Volume
):

    input_data = create_input_features(
        Prev_Open, Prev_High, Prev_Low, Prev_Close, Prev_Volume,
        Open, High, Low, Close, Volume
    )

    direction_pred = clf_direction.predict(input_data)[0]
    direction_prob = clf_direction.predict_proba(input_data)[0]

    prob_down = direction_prob[0] * 100
    prob_up = direction_prob[1] * 100

    if direction_pred == 1:
        direction = "UP"
        icon = "▲"
        signal_class = "up"
        direction_text = "Price is expected to increase"
    else:
        direction = "DOWN"
        icon = "▼"
        signal_class = "down"
        direction_text = "Price is expected to decrease"

    predicted_delta = reg_delta.predict(input_data)[0]

    tomorrow_prediction = Close + predicted_delta

    after_tomorrow_input = create_input_features(
        Prev_Open=Open,
        Prev_High=High,
        Prev_Low=Low,
        Prev_Close=Close,
        Prev_Volume=Volume,

        Open=Close,
        High=max(High, tomorrow_prediction),
        Low=min(Low, tomorrow_prediction),
        Close=tomorrow_prediction,
        Volume=Volume
    )

    after_tomorrow_delta = reg_delta.predict(after_tomorrow_input)[0]
    after_tomorrow_prediction = tomorrow_prediction + after_tomorrow_delta

    change_percent = (predicted_delta / Close) * 100

    main_result = f"""
    <div class="main-price">
        <div class="location">XAU / Gold Market</div>

        <div class="price">{tomorrow_prediction:.2f}</div>

        <div class="desc">Predicted Tomorrow Closing Price</div>

        <div class="signal {signal_class}">
            {icon} {direction}
        </div>

        <div class="change-box">
            Predicted Delta: {predicted_delta:+.2f} USD ({change_percent:+.2f}%)
        </div>

        <div class="change-box">
            {direction_text}
        </div>
    </div>
    """

    forecast = f"""
    <div class="forecast-card">
        <div class="forecast-title">Daily Forecast</div>

        <div class="forecast-row">
            <span>Today Close</span>
            <div class="forecast-price">{Close:.2f}</div>
            <b>Current</b>
        </div>

        <div class="forecast-row">
            <span>Tomorrow Close</span>
            <div class="forecast-price">{tomorrow_prediction:.2f}</div>
            <b>Real Predict</b>
        </div>

        <div class="forecast-row">
            <span>After Tomorrow</span>
            <div class="forecast-price">{after_tomorrow_prediction:.2f}</div>
            <b>Real Predict</b>
        </div>

        <div class="forecast-row">
            <span>Predicted Delta</span>
            <div class="forecast-price">{predicted_delta:+.2f}</div>
            <b>Regression</b>
        </div>

        <div class="forecast-row">
            <span>Direction</span>
            <div class="forecast-price">{direction}</div>
            <b>Logistic Regression</b>
        </div>

        <div class="forecast-row">
            <span>UP Probability</span>
            <div class="forecast-price">{prob_up:.2f}%</div>
            <b>Class 1</b>
        </div>

        <div class="forecast-row">
            <span>DOWN Probability</span>
            <div class="forecast-price">{prob_down:.2f}%</div>
            <b>Class 0</b>
        </div>

        <div class="forecast-row">
            <span>Direction Accuracy</span>
            <div class="forecast-price">{direction_accuracy * 100:.2f}%</div>
            <b>Test Score</b>
        </div>

        <div class="forecast-row">
            <span>Delta MAE</span>
            <div class="forecast-price">{delta_mae:.2f}</div>
            <b>Regression</b>
        </div>

        <div class="forecast-row">
            <span>Naive MAE</span>
            <div class="forecast-price">{naive_delta_mae:.2f}</div>
            <b>Baseline</b>
        </div>

        <div class="model-note">
            Direction is predicted using Logistic Regression. Real price prediction is calculated using a regression model for Target_Delta.
        </div>
    </div>
    """

    return (
        main_result,
        gr.update(visible=True),
        make_price_chart(),
        forecast
    )

In [12]:
css = """
.gradio-container {
    background: linear-gradient(180deg, #111827, #1e3a5f, #263b63);
    color: white;
    font-family: Arial, sans-serif;
}

#app-title {
    text-align: center;
    font-size: 38px;
    font-weight: bold;
    color: white;
    margin-bottom: 5px;
}

#app-subtitle {
    text-align: center;
    color: #d1d5db;
    margin-bottom: 25px;
    font-size: 18px;
}

.main-card {
    background: rgba(255, 255, 255, 0.12);
    border-radius: 28px;
    padding: 25px;
    box-shadow: 0px 8px 25px rgba(0,0,0,0.35);
    backdrop-filter: blur(10px);
}

.input-card {
    background: rgba(255, 255, 255, 0.10);
    border-radius: 22px;
    padding: 20px;
}

.main-price {
    text-align: center;
    padding: 10px;
}

.location {
    font-size: 26px;
    color: #f3f4f6;
}

.price {
    font-size: 76px;
    font-weight: 300;
    color: white;
    margin-top: 10px;
}

.desc {
    font-size: 18px;
    color: #d1d5db;
}

.forecast-card {
    background: rgba(255, 255, 255, 0.13);
    border-radius: 24px;
    padding: 22px;
    margin-top: 20px;
}

.forecast-title {
    font-size: 22px;
    font-weight: bold;
    margin-bottom: 18px;
    color: white;
}

.forecast-row {
    display: grid;
    grid-template-columns: 170px 1fr 130px;
    align-items: center;
    gap: 15px;
    padding: 14px 0;
    border-bottom: 1px solid rgba(255,255,255,0.15);
    font-size: 18px;
    color: white;
}

.forecast-price {
    font-size: 24px;
    font-weight: bold;
    color: #f3f4f6;
}

button {
    background: #800020 !important;
    color: white !important;
    font-weight: bold !important;
    border-radius: 16px !important;
    height: 48px;
    border: none !important;
}

button:hover {
    background: #991b1b !important;
    color: white !important;
}

input, textarea {
    border-radius: 14px !important;
}

.signal {
    display: inline-block;
    margin-top: 18px;
    padding: 10px 24px;
    border-radius: 999px;
    font-size: 22px;
    font-weight: bold;
}

.up {
    background: rgba(34, 197, 94, 0.18);
    color: #22c55e;
    border: 1px solid rgba(34, 197, 94, 0.45);
}

.down {
    background: rgba(239, 68, 68, 0.18);
    color: #ef4444;
    border: 1px solid rgba(239, 68, 68, 0.45);
}

.stable {
    background: rgba(148, 163, 184, 0.18);
    color: #cbd5e1;
    border: 1px solid rgba(148, 163, 184, 0.45);
}

.change-box {
    margin-top: 14px;
    font-size: 18px;
    color: #e5e7eb;
}

.model-note {
    margin-top: 18px;
    font-size: 14px;
    color: #cbd5e1;
    opacity: 0.85;
    line-height: 1.5;
}

footer {
    display: none !important;
}
"""

In [ ]:
with gr.Blocks(css=css, title="XAU Forecast App") as demo:

    gr.HTML("""
    <div id="app-title">XAU Price Forecast App</div>
    <div id="app-subtitle">Machine Learning Demo Application</div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group(elem_classes="input-card"):
                gr.Markdown("### Previous Day Values")

                Prev_Open = gr.Number(label="Previous Open", value=2280)
                Prev_High = gr.Number(label="Previous High", value=2310)
                Prev_Low = gr.Number(label="Previous Low", value=2260)
                Prev_Close = gr.Number(label="Previous Close", value=2290)
                Prev_Volume = gr.Number(label="Previous Volume", value=9500)

                gr.Markdown("### Today Values")

                Open = gr.Number(label="Today Open", value=2300)
                High = gr.Number(label="Today High", value=2320)
                Low = gr.Number(label="Today Low", value=2280)
                Close = gr.Number(label="Today Close", value=2310)
                Volume = gr.Number(label="Today Volume", value=10000)

                predict_btn = gr.Button("Predict")

        with gr.Column(scale=2):
            with gr.Group(elem_classes="main-card"):

                main_output = gr.HTML("""
                <div class="main-price">
                    <div class="location">XAU / Gold Market</div>
                    <div class="price">----</div>
                    <div class="desc">Predicted Tomorrow Closing Price</div>
                </div>
                """)

                with gr.Column(visible=False) as result_block:

                    gr.Markdown("### Average XAU Price by Year")

                    price_chart_output = gr.Plot(
                        label="Average XAU Price by Year"
                    )

                    forecast_output = gr.HTML()

    predict_btn.click(
        fn=predict_app,
        inputs=[
            Prev_Open, Prev_High, Prev_Low, Prev_Close, Prev_Volume,
            Open, High, Low, Close, Volume
        ],
        outputs=[
            main_output,
            result_block,
            price_chart_output,
            forecast_output
        ]
    )

demo.launch(
    share=True,
    debug=True,
    show_api=False
)



/tmp/ipykernel_1366/3858102285.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="XAU Forecast App") as demo:
/tmp/ipykernel_1366/3858102285.py:64: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e4f85174397a0968e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
